In [1]:
!pip install -U google-genai psycopg[binary] pgvector

Defaulting to user installation because normal site-packages is not writeable
  Using cached google_genai-2.28.0-py3-none-any.whl.metadata (56 kB)
Using cached google_genai-2.28.0-py3-none-any.whl (1.2 MB)
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.7 MB 2.4 MB/s eta 0:00:02
   ----------- ---------------------------- 1.0/3.7 MB 2.8 MB/s eta 0:00:01
   ----------------- ---------------------- 1.6/3.7 MB 2.6 MB/s eta 0:00:01
   ---------------------- ----------------- 2.1/3.7 MB 2.6 MB/s eta 0:00:01
   ---------------------------- ----------- 2.6/3.7 MB 2.6 MB/s eta 0:00:01
   ------------------------------- -------- 2.9/3.7 MB 2.6 MB/s eta 0:00:01
   ------------------------------------- -- 3.4/3.7 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 2.5 MB/s  0:00:01

  Attempting uninstall: psycopg-binary

    Found existing installation: psycopg-binary 3.3.4

    Uninstalling psycopg

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.149.0 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.28.0 which is incompatible.
langchain-google-genai 4.2.2 requires google-genai<2.0.0,>=1.65.0, but you have google-genai 2.28.0 which is incompatible.

[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg

from google import genai
from google.genai import types

print("All imports successful!")

All imports successful!


In [3]:
import sys

print("Python:", sys.version)

try:
    import google.genai
    print("google-genai:", google.genai.__version__)
except Exception as e:
    print("google-genai version unavailable:", e)

print("psycopg:", psycopg.__version__)

Python: 3.11.6 (tags/v3.11.6:8b6ee5b, Oct  2 2023, 14:57:12) [MSC v.1935 64 bit (AMD64)]
google-genai: 2.28.0
psycopg: 3.3.6


In [4]:
with open("api_key.txt", "r") as f:
    GEMINI_API_KEY = f.read().strip()

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client ready")

Gemini client ready


In [5]:
document = """
Artificial Intelligence is a field of computer science that focuses on
creating systems that can perform tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence. It allows computers
to learn patterns from data without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses neural networks with
multiple layers. Deep Learning is commonly used for image recognition,
natural language processing, speech recognition, and recommendation systems.

Generative AI is a branch of Artificial Intelligence that can generate new
content such as text, images, audio, video, and computer code.

Large Language Models are Generative AI models trained on very large amounts
of text. They can understand and generate human-like language.

Retrieval Augmented Generation, commonly called RAG, combines information
retrieval with a language model. Instead of relying only on the knowledge
stored inside the model, RAG retrieves relevant information from an external
knowledge source and provides that information to the language model.
"""

print(document)


Artificial Intelligence is a field of computer science that focuses on
creating systems that can perform tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence. It allows computers
to learn patterns from data without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses neural networks with
multiple layers. Deep Learning is commonly used for image recognition,
natural language processing, speech recognition, and recommendation systems.

Generative AI is a branch of Artificial Intelligence that can generate new
content such as text, images, audio, video, and computer code.

Large Language Models are Generative AI models trained on very large amounts
of text. They can understand and generate human-like language.

Retrieval Augmented Generation, commonly called RAG, combines information
retrieval with a language model. Instead of relying only on the knowledge
stored inside the model, RAG retr

In [6]:
def chunk_text(text, chunk_size=50, overlap=10):
    words = text.split()

    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [7]:
chunks = chunk_text(
    document,
    chunk_size=50,
    overlap=10
)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [8]:
for i, chunk in enumerate(chunks, start=1):
    print(f"\n========== CHUNK {i} ==========")
    print(chunk)


========== CHUNK 1 ==========
Artificial Intelligence is a field of computer science that focuses on creating systems that can perform tasks that normally require human intelligence. Machine Learning is a subset of Artificial Intelligence. It allows computers to learn patterns from data without being explicitly programmed for every task. Deep Learning is a subset

========== CHUNK 2 ==========
explicitly programmed for every task. Deep Learning is a subset of Machine Learning that uses neural networks with multiple layers. Deep Learning is commonly used for image recognition, natural language processing, speech recognition, and recommendation systems. Generative AI is a branch of Artificial Intelligence that can generate new content such

========== CHUNK 3 ==========
branch of Artificial Intelligence that can generate new content such as text, images, audio, video, and computer code. Large Language Models are Generative AI models trained on very large amounts of text. They can unders

In [9]:
chunk_data = []

for i, chunk in enumerate(chunks, start=1):

    chunk_data.append({
        "chunk_id": i,
        "content": chunk
    })

for item in chunk_data:
    print("Chunk ID:", item["chunk_id"])
    print("Content:", item["content"])
    print()

Chunk ID: 1
Content: Artificial Intelligence is a field of computer science that focuses on creating systems that can perform tasks that normally require human intelligence. Machine Learning is a subset of Artificial Intelligence. It allows computers to learn patterns from data without being explicitly programmed for every task. Deep Learning is a subset

Chunk ID: 2
Content: explicitly programmed for every task. Deep Learning is a subset of Machine Learning that uses neural networks with multiple layers. Deep Learning is commonly used for image recognition, natural language processing, speech recognition, and recommendation systems. Generative AI is a branch of Artificial Intelligence that can generate new content such

Chunk ID: 3
Content: branch of Artificial Intelligence that can generate new content such as text, images, audio, video, and computer code. Large Language Models are Generative AI models trained on very large amounts of text. They can understand and generate human-like

In [10]:
EMBEDDING_MODEL = "gemini-embedding-001"

In [11]:
EMBEDDING_MODEL = "gemini-embedding-001"


def create_embedding(text):

    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            output_dimensionality=768
        )
    )

    return result.embeddings[0].values

In [12]:
for item in chunk_data:

    item["embedding"] = create_embedding(
        item["content"]
    )

print("Embedding created for all chunks")

Embedding created for all chunks


In [15]:
print("Number of chunks:", len(chunk_data))
print("Embedding size:", (chunk_data[0]["embedding"]))

Number of chunks: 4
Embedding size: [-0.0030833825, -0.0069081835, 0.0038206135, -0.043873742, -0.04521386, 0.012408289, 0.0052434755, 0.0039571756, 0.024366165, 0.009434473, 0.009354937, 0.007751212, -0.0035054164, 0.030671438, 0.11812465, -0.0056800507, -0.012361708, 0.00163408, 0.004675635, -0.011007915, 0.005375174, -0.011292351, 0.0051390897, -0.02793952, -0.036563534, -0.011671979, -0.0074121114, 0.03178681, 0.02663796, -0.0072813635, 0.0008577999, 0.008023548, 0.001543923, 0.027657859, -0.011690642, 0.0136082675, 0.013914676, -0.030905388, 0.026479958, 0.02460773, -0.0021684915, -0.01802804, -0.024649607, -0.0043047457, 0.011365734, 0.0014927106, 0.003979824, -0.035910916, -0.026913464, 0.018739112, -0.007935674, 0.0018876435, 0.0074181417, -0.17536518, -0.008677751, 0.003511014, -0.009258673, -0.013454268, 0.019025175, -0.0023626045, -0.016501216, 0.028873166, -0.015228492, -0.036227997, -0.012400282, -0.025085667, 0.013849486, -0.0015636098, 0.0018283462, 0.013552797, 0.004790